# 🚇 MetrôBot SP 2.0 — Desafio (Linhas 1-Azul, 2-Verde e 3-Vermelha)

Giovanni Pinheiro Bonifatto - RA1635819

## Como rodar
1. **Colab:** Ambiente de execução → **Executar tudo**. **VS Code:** *Run All* (extensões Python + Jupyter).
2. **Com Llama (opcional):** guarde a chave do Groq em *Secrets* do Colab (nome `GROQ_API_KEY`) ou em um
   arquivo `.env` (`GROQ_API_KEY=...`).
3. **Sem chave / sem internet:** troque `PROVEDOR = "offline"` na célula 2.

In [ ]:
# Célula 1 — instalação
%pip install -q groq ollama ipywidgets python-dotenv

In [ ]:
# Célula 2 — configuração e função única de LLM
import os, json, re, unicodedata
from collections import deque
from itertools import product

PROVEDOR = "groq"                  # "groq" (nuvem), "ollama" (local) ou "offline" (sem LLM)
MODELO_GROQ = "openai/gpt-oss-20b"
MODELO_OLLAMA = "llama3.2"


def obter_chave_groq():
    """Busca a chave SEM escrevê-la no código: Colab Secrets → .env → variável de ambiente."""
    try:
        from google.colab import userdata
        for nome_secret in ("GROQ_API_KEY", "GROQ"):      # aceita os dois nomes de secret
            try:
                chave = userdata.get(nome_secret)
                if chave:
                    return chave
            except Exception:
                continue
    except Exception:
        pass
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except Exception:
        pass
    return os.environ.get("GROQ_API_KEY")


def chamar_llm(mensagens, modo_json=False):
    """Envia mensagens ao Llama e devolve o texto da resposta."""
    if PROVEDOR == "groq":
        from groq import Groq
        cliente = Groq(api_key=obter_chave_groq())
        extras = {"response_format": {"type": "json_object"}} if modo_json else {}
        resposta = cliente.chat.completions.create(
            model=MODELO_GROQ, messages=mensagens, temperature=0, **extras)
        return resposta.choices[0].message.content
    elif PROVEDOR == "ollama":
        import ollama
        extras = {"format": "json"} if modo_json else {}
        resposta = ollama.chat(model=MODELO_OLLAMA, messages=mensagens,
                               options={"temperature": 0}, **extras)
        return resposta["message"]["content"]
    else:
        raise RuntimeError("Modo offline: nenhum LLM configurado.")

In [ ]:
# Célula 3 — teste de conexão (não quebra o notebook se não houver chave/internet)
if PROVEDOR == "offline":
    print("Modo offline: o MetrôBot vai funcionar sem LLM.")
else:
    try:
        print(chamar_llm([{"role": "user",
                           "content": "Em uma frase curta, dê boas-vindas aos passageiros do metrô de São Paulo."}]))
    except Exception as erro:
        print(f"⚠️ LLM indisponível ({type(erro).__name__}). O MetrôBot vai usar o modo offline automaticamente.")

Bem-vindos ao metrô de São Paulo!


In [ ]:
# Célula 4 — dados das linhas
LINHAS = {
    "Linha 1-Azul": [
        "Tucuruvi", "Parada Inglesa", "Jardim São Paulo", "Santana",
        "Carandiru", "Portuguesa-Tietê", "Armênia", "Tiradentes", "Luz",
        "São Bento", "Sé", "Japão-Liberdade", "São Joaquim", "Vergueiro",
        "Paraíso", "Ana Rosa", "Vila Mariana", "Santa Cruz",
        "Praça da Árvore", "Saúde", "São Judas", "Conceição", "Jabaquara",
    ],
    "Linha 2-Verde": [
        "Vila Madalena", "Sumaré", "Clínicas", "Consolação", "Trianon-Masp",
        "Brigadeiro", "Paraíso", "Ana Rosa", "Chácara Klabin",
        "Santos-Imigrantes", "Alto do Ipiranga", "Sacomã", "Tamanduateí",
        "Vila Prudente",
    ],
    "Linha 3-Vermelha": [
        "Palmeiras-Barra Funda", "Marechal Deodoro", "Santa Cecília",
        "República", "Anhangabaú", "Sé", "Pedro II", "Brás",
        "Bresser-Mooca", "Belém", "Tatuapé", "Carrão", "Penha",
        "Vila Matilde", "Guilhermina-Esperança", "Patriarca-Vila Ré",
        "Artur Alvim", "Corinthians-Itaquera",
    ],
}

CORES = {"Linha 1-Azul": "#1e88e5", "Linha 2-Verde": "#2e7d32",
         "Linha 3-Vermelha": "#d32f2f"}

# Uma estação = um nó, mesmo que pertença a duas linhas (Sé, Paraíso e Ana Rosa aparecem 2x nos dados).
ESTACOES = list(dict.fromkeys(e for est in LINHAS.values() for e in est))
assert len(ESTACOES) == 52, "esperado: 52 estações únicas"

# Locais conhecidos (local → estação MAIS PRÓXIMA dentro do nosso modelo de 3 linhas).
LOCAIS = {
    # --- Linha 1-Azul ---
    "Shopping Metrô Tucuruvi": "Tucuruvi",
    "Terminal Rodoviário Tietê": "Portuguesa-Tietê",
    "Museu de Arte Sacra": "Tiradentes",
    "Pinacoteca": "Luz",
    "Museu da Língua Portuguesa": "Luz",
    "Mosteiro de São Bento": "São Bento",
    "Rua 25 de Março": "São Bento",
    "Catedral da Sé": "Sé",
    "Bairro da Liberdade": "Japão-Liberdade",
    "Centro Cultural São Paulo": "Vergueiro",
    "Shopping Metrô Santa Cruz": "Santa Cruz",
    "Universidade São Judas": "São Judas",
    "Terminal Rodoviário Jabaquara": "Jabaquara",
    # --- Linha 2-Verde ---
    "MASP": "Trianon-Masp",
    "Parque Trianon": "Trianon-Masp",
    "Hospital das Clínicas": "Clínicas",
    "Beco do Batman": "Vila Madalena",
    # --- Linha 3-Vermelha ---
    "Nubank Parque": "Palmeiras-Barra Funda",
    "Terminal Rodoviário Barra Funda": "Palmeiras-Barra Funda",
    "Praça da República": "República",
    "Theatro Municipal": "Anhangabaú",
    "Neo Química Arena": "Corinthians-Itaquera",
}
assert all(est in ESTACOES for est in LOCAIS.values()), "local apontando para estação inexistente"

In [ ]:
# Célula 5 — R1: grafo das 3 linhas
def construir_grafo_multilinhas(linhas):
    """Retorna (grafo, linhas_do_trecho).
    grafo: {estacao: [vizinhas sem repetição]}
    linhas_do_trecho: {(a, b): {nomes das linhas}} — guarda (a,b) e (b,a)
    """
    grafo = {}
    linhas_do_trecho = {}
    for nome_linha, estacoes in linhas.items():
        for i in range(len(estacoes) - 1):
            a, b = estacoes[i], estacoes[i + 1]
            grafo.setdefault(a, [])
            grafo.setdefault(b, [])
            if b not in grafo[a]:
                grafo[a].append(b)
            if a not in grafo[b]:
                grafo[b].append(a)
            linhas_do_trecho.setdefault((a, b), set()).add(nome_linha)
            linhas_do_trecho.setdefault((b, a), set()).add(nome_linha)
    return grafo, linhas_do_trecho


def construir_grafo_ativo(linhas, paralisadas=()):
    """Grafo só com as linhas em operação. Estações servidas apenas por linhas paralisadas ficam isoladas."""
    ativas = {nome: est for nome, est in linhas.items() if nome not in paralisadas}
    grafo, trechos = construir_grafo_multilinhas(ativas)
    for e in ESTACOES:
        grafo.setdefault(e, [])
    return grafo, trechos


GRAFO, LINHAS_DO_TRECHO = construir_grafo_multilinhas(LINHAS)

print("Total de estações:", len(GRAFO))
print("Vizinhas da Sé:", GRAFO["Sé"])
print("Vizinhas do Paraíso:", GRAFO["Paraíso"])
print("Linhas do trecho Paraíso–Ana Rosa:", sorted(LINHAS_DO_TRECHO[("Paraíso", "Ana Rosa")]))

Total de estações: 52
Vizinhas da Sé: ['São Bento', 'Japão-Liberdade', 'Anhangabaú', 'Pedro II']
Vizinhas do Paraíso: ['Vergueiro', 'Ana Rosa', 'Brigadeiro']
Linhas do trecho Paraíso–Ana Rosa: ['Linha 1-Azul', 'Linha 2-Verde']


In [ ]:
# Célula 6 — R2: BFS e DFS
def reconstruir_caminho(pai, destino):
    """Puxa o 'fio de Ariadne': do destino até a origem, depois inverte."""
    caminho = []
    atual = destino
    while atual is not None:
        caminho.append(atual)
        atual = pai[atual]
    return list(reversed(caminho))


def bfs(grafo, origem, destino, bloqueadas=()):
    """Busca em Largura: fila (FIFO). Garante o menor número de paradas."""
    if origem in bloqueadas or destino in bloqueadas:
        return None, []
    fila = deque([origem])
    pai = {origem: None}
    ordem_visita = []
    while fila:
        atual = fila.popleft()
        ordem_visita.append(atual)
        if atual == destino:
            return reconstruir_caminho(pai, destino), ordem_visita
        for vizinho in grafo[atual]:
            if vizinho not in pai and vizinho not in bloqueadas:
                pai[vizinho] = atual
                fila.append(vizinho)
    return None, ordem_visita


def dfs(grafo, origem, destino, bloqueadas=()):
    """Busca em Profundidade: vai fundo no 1º vizinho; se não achar, volta (backtracking)."""
    if origem in bloqueadas or destino in bloqueadas:
        return None, []
    visitados = set()
    ordem_visita = []

    def explorar(atual, caminho):
        visitados.add(atual)
        ordem_visita.append(atual)
        if atual == destino:
            return caminho
        for vizinho in grafo[atual]:
            if vizinho not in visitados and vizinho not in bloqueadas:
                resultado = explorar(vizinho, caminho + [vizinho])
                if resultado:
                    return resultado
        return None

    return explorar(origem, [origem]), ordem_visita

In [ ]:
# Célula 7 — baldeações
def linhas_por_trecho(caminho, linhas_do_trecho):
    """Escolhe UMA linha para cada trecho do caminho, minimizando as trocas.
    Regra gulosa: enquanto a linha atual serve para o próximo trecho, continue nela; quando não serve,
    troque para a linha que atende o maior número de trechos seguidos (isso é ótimo para minimizar trocas)."""
    trechos = [linhas_do_trecho[(caminho[i], caminho[i + 1])] for i in range(len(caminho) - 1)]

    def alcance(i, linha):
        n = 0
        while i + n < len(trechos) and linha in trechos[i + n]:
            n += 1
        return n

    escolhidas = []
    atual = None
    for i, opcoes in enumerate(trechos):
        if atual is None or atual not in opcoes:
            atual = max(sorted(opcoes), key=lambda l: alcance(i, l))
        escolhidas.append(atual)
    return escolhidas


def contar_baldeacoes(caminho, linhas_do_trecho):
    """Retorna (quantidade, lista de (estacao, linha_nova)). A baldeação é anotada na estação onde a troca ocorre."""
    if not caminho or len(caminho) < 2:
        return 0, []
    escolhidas = linhas_por_trecho(caminho, linhas_do_trecho)
    baldeacoes = [(caminho[i], escolhidas[i]) for i in range(1, len(escolhidas))
                  if escolhidas[i] != escolhidas[i - 1]]
    return len(baldeacoes), baldeacoes


def segmentos_do_caminho(caminho, linhas_do_trecho):
    """Quebra o caminho em pedaços de uma mesma linha: [(linha, estacao_inicio, estacao_fim, n_paradas)]."""
    if not caminho or len(caminho) < 2:
        return []
    escolhidas = linhas_por_trecho(caminho, linhas_do_trecho)
    segmentos = []
    for i, linha in enumerate(escolhidas):
        if segmentos and segmentos[-1][0] == linha:
            segmentos[-1][2] = caminho[i + 1]
            segmentos[-1][3] += 1
        else:
            segmentos.append([linha, caminho[i], caminho[i + 1], 1])
    return [tuple(s) for s in segmentos]

In [ ]:
# Célula 8 — Tabela Verdade
def pode_baldear(P, I, Q, R):
    """P: estação aberta | I: é estação de integração | Q: passageiro precisa de acessibilidade
    | R: elevador funcionando.   pode_baldear ≡ P ∧ I ∧ (¬Q ∨ R)"""
    return P and I and ((not Q) or R)


def tabela_verdade(funcao, nomes, expressao):
    """Gera a tabela-verdade de qualquer função booleana por código."""
    print(" | ".join(f"{n:^7}" for n in nomes) + f" | {expressao}")
    print("-" * (10 * len(nomes) + len(expressao)))
    for valores in product([True, False], repeat=len(nomes)):
        print(" | ".join(f"{str(v):^7}" for v in valores) + f" | {funcao(*valores)}")


tabela_verdade(pode_baldear, ["P", "I", "Q", "R"], "P ∧ I ∧ (¬Q ∨ R)")

   P    |    I    |    Q    |    R    | P ∧ I ∧ (¬Q ∨ R)
--------------------------------------------------------
 True   |  True   |  True   |  True   | True
 True   |  True   |  True   |  False  | False
 True   |  True   |  False  |  True   | True
 True   |  True   |  False  |  False  | True
 True   |  False  |  True   |  True   | False
 True   |  False  |  True   |  False  | False
 True   |  False  |  False  |  True   | False
 True   |  False  |  False  |  False  | False
 False  |  True   |  True   |  True   | False
 False  |  True   |  True   |  False  | False
 False  |  True   |  False  |  True   | False
 False  |  True   |  False  |  False  | False
 False  |  False  |  True   |  True   | False
 False  |  False  |  True   |  False  | False
 False  |  False  |  False  |  True   | False
 False  |  False  |  False  |  False  | False


In [ ]:
# Célula 9 — R3: lógica de primeira ordem
# Fatos, regras R1–R7 e motor de inferência
# ---------------------------------------------------------------------------

from datetime import datetime, timezone, timedelta

# O Colab roda em UTC: datetime.now() sozinho erra 3 horas para quem está em São Paulo.
try:
    from zoneinfo import ZoneInfo
    FUSO_SP = ZoneInfo("America/Sao_Paulo")
except Exception:                                   # sem base de fusos: UTC-3 fixo (SP não tem horário de verão)
    FUSO_SP = timezone(timedelta(hours=-3))


RELOGIO_FIXO = None    # só para testes: "HH:MM" força o "agora" (None = relógio real)


def agora_sp():
    """Data/hora atual em São Paulo."""
    agora = datetime.now(FUSO_SP)
    if RELOGIO_FIXO:
        h, m = map(int, RELOGIO_FIXO.split(":"))
        agora = agora.replace(hour=h, minute=m, second=0, microsecond=0)
    return agora


# Operação do metrô: antes da abertura o metrô está FECHADO e não há rota.
ABERTURA_MIN = 280
ABERTURA_METRO = f"{ABERTURA_MIN // 60:02d}:{ABERTURA_MIN % 60:02d}"
SITUACOES_SEM_OPERACAO = {"fechado"}


def validar_horario(valor):
    """Normaliza um horário para "HH:MM" ou devolve None se for inválido.
    Aceita "8:05", "08:05", "18h30", "18h". Rejeita "25:00", "às 8", etc.
    Serve de guardrail para o que vem do LLM ou do usuário."""
    if valor is None:
        return None
    if hasattr(valor, "hour") and hasattr(valor, "minute"):
        return f"{valor.hour:02d}:{valor.minute:02d}"
    if not isinstance(valor, str):
        return None
    m = re.fullmatch(r"\s*(\d{1,2})\s*[:hH]\s*(\d{2})?\s*", valor)
    if not m:
        return None
    hora, minuto = int(m.group(1)), int(m.group(2) or 0)
    if 0 <= hora <= 23 and 0 <= minuto <= 59:
        return f"{hora:02d}:{minuto:02d}"
    return None

def fatos_base():
    """
    Fatos fixos do mundo:
    - linhas existentes;
    - estações;
    - relação estação -> linha;
    - locais próximos às estações.
    """
    fatos = set()

    for linha, estacoes in LINHAS.items():
        fatos.add(("linha", linha))

        for estacao in estacoes:
            fatos.add(("estacao", estacao))
            fatos.add(("pertence", estacao, linha))

    for local, estacao in LOCAIS.items():
        fatos.add(("proximo_de", local, estacao))

    return fatos

def consultar(fatos, predicado):
    """
    Retorna os argumentos de todos os fatos de determinado predicado.

    Exemplo:
        consultar(fatos, "destino")

    pode retornar:
        [("Luz",)]
    """
    return [
        fato[1:]
        for fato in fatos
        if fato[0] == predicado
    ]

def r_origem(fatos):
    """
    R1:
    Se o usuário está em um local próximo a uma estação,
    essa estação é considerada a origem.

    Também aceita diretamente uma estação informada pelo usuário.
    """

    novos = set()

    for (local,) in consultar(fatos, "usuario_esta_em"):
        for (local_proximo, estacao) in consultar(fatos, "proximo_de"):
            if local_proximo == local:
                novos.add(("origem", estacao))

    for (estacao,) in consultar(fatos, "usuario_esta_na_estacao"):
        novos.add(("origem", estacao))

    return novos

def r_destino(fatos):
    """
    R2:
    Se o usuário quer ir para um local próximo a uma estação,
    essa estação é considerada o destino.

    Também aceita diretamente uma estação informada pelo usuário.
    """

    novos = set()

    for (local,) in consultar(fatos, "usuario_quer_ir"):
        for (local_proximo, estacao) in consultar(fatos, "proximo_de"):
            if local_proximo == local:
                novos.add(("destino", estacao))

    for (estacao,) in consultar(fatos, "usuario_quer_ir_estacao"):
        novos.add(("destino", estacao))

    return novos

def r_bloqueio(fatos):
    """
    R3:
    Uma estação fechada é considerada bloqueada.
    """

    return {
        ("bloqueada", estacao)
        for (estacao,) in consultar(fatos, "fechada")
    }

def r_acessibilidade(fatos):
    """
    R4:
    Se o usuário precisa de acessibilidade e o elevador
    de uma estação está em manutenção, a estação é
    considerada inacessível.
    """

    if not consultar(fatos, "precisa_acessibilidade"):
        return set()

    return {
        ("inacessivel", estacao)
        for (estacao,) in consultar(
            fatos,
            "elevador_em_manutencao"
        )
    }

def r_alerta(fatos):
    """
    R5:
    Se a origem ou o destino for uma estação inacessível,
    gera um alerta para esse ponto da viagem.
    """

    novos = set()

    inacessiveis = {
        estacao
        for (estacao,) in consultar(
            fatos,
            "inacessivel"
        )
    }

    for papel in ("origem", "destino"):

        for (estacao,) in consultar(
            fatos,
            papel
        ):

            if estacao in inacessiveis:
                novos.add(
                    ("alerta", papel, estacao)
                )

    return novos

def r_integracao(fatos):
    """
    R6:
    Uma estação é uma integração quando pertence a
    pelo menos duas linhas diferentes.

    A integração é deduzida automaticamente pelo motor.
    """

    pertence = consultar(
        fatos,
        "pertence"
    )

    novos = set()

    for (estacao1, linha1) in pertence:

        for (estacao2, linha2) in pertence:

            if (
                estacao1 == estacao2
                and linha1 != linha2
            ):
                novos.add(
                    ("integracao", estacao1)
                )

    return novos

def r_linha_paralisada(fatos):
    """
    Bônus:
    Uma estação atendida SOMENTE por linhas paralisadas
    fica bloqueada.

    Estações de integração continuam abertas quando
    pelo menos uma das linhas que as atende não está paralisada.

    Formalmente:

    ∀e (
        (∃l pertence(e,l))
        ∧
        ∀l (pertence(e,l) → paralisada(l))
        →
        bloqueada(e)
    )
    """

    paralisadas = {
        linha
        for (linha,) in consultar(
            fatos,
            "paralisada"
        )
    }

    if not paralisadas:
        return set()

    linhas_da_estacao = {}

    for estacao, linha in consultar(
        fatos,
        "pertence"
    ):
        linhas_da_estacao.setdefault(
            estacao,
            set()
        ).add(linha)

    return {
        ("bloqueada", estacao)
        for estacao, linhas in linhas_da_estacao.items()
        if linhas <= paralisadas
    }

def obter_situacao_metro(hora_atual=None):
    """
    Determina a situação estimada do metrô de acordo com o horário.

    Se nenhum horário for informado, utiliza o horário atual
    do computador.

    Aceita:
        - datetime.time
        - string no formato "HH:MM"
    """

    # Se o usuário não informou horário,
    # utiliza o horário atual do computador.
    if hora_atual is None:
        hora_atual = agora_sp().time()

    # Permite receber horário como texto.
    if isinstance(hora_atual, str):
        normalizado = validar_horario(hora_atual)

        if normalizado is None:
            raise ValueError(
                f"Horário inválido: {hora_atual!r} "
                "(use HH:MM, entre 00:00 e 23:59)"
            )

        hora_atual = datetime.strptime(
            normalizado,
            "%H:%M"
        ).time()

    minutos = (
        hora_atual.hour * 60
        + hora_atual.minute
    )

    # ---------------------------------------------------------------
    # METRÔ FECHADO
    # ---------------------------------------------------------------

    if minutos < ABERTURA_MIN:
        return {
            "estado": "fechado",
            "lotacao": "sem movimento"
        }

    # ---------------------------------------------------------------
    # PRESTES A ABRIR
    # ---------------------------------------------------------------

    elif minutos < 330:
        return {
            "estado": "prestes a abrir",
            "lotacao": "movimento muito baixo"
        }

    # ---------------------------------------------------------------
    # ACABOU DE ABRIR
    # ---------------------------------------------------------------

    elif minutos < 390:
        return {
            "estado": "acabou de abrir",
            "lotacao": "movimento baixo"
        }

    # ---------------------------------------------------------------
    # PICO DA MANHÃ
    # ---------------------------------------------------------------

    elif minutos <= 570:
        return {
            "estado": "horário de pico",
            "lotacao": "provavelmente cheio"
        }

    # ---------------------------------------------------------------
    # HORÁRIO INTERMEDIÁRIO
    # ---------------------------------------------------------------

    elif minutos < 990:
        return {
            "estado": "horário intermediário",
            "lotacao": "movimento moderado"
        }

    # ---------------------------------------------------------------
    # PICO DA TARDE/NOITE
    # ---------------------------------------------------------------

    elif minutos <= 1170:
        return {
            "estado": "horário de pico",
            "lotacao": "provavelmente cheio"
        }

    # ---------------------------------------------------------------
    # HORÁRIO NOTURNO
    # ---------------------------------------------------------------

    elif minutos <= 1380:
        return {
            "estado": "horário noturno",
            "lotacao": "movimento moderado"
        }

    # ---------------------------------------------------------------
    # PRESTES A FECHAR
    # ---------------------------------------------------------------

    else:
        return {
            "estado": "prestes a fechar",
            "lotacao": "movimento baixo"
        }


def r_situacao_horario(fatos):
    """
    R7:
    Determina a situação do metrô com base no horário informado.

    O planejador sempre informa o horário considerado
    (o do usuário ou, se ele não informou, o horário atual de São Paulo).
    """

    horario = next(
        (
            valor
            for (valor,) in consultar(
                fatos,
                "horario_considerado"
            )
        ),
        None
    )

    situacao = obter_situacao_metro(
        horario
    )

    return {
        (
            "situacao_metro",
            situacao["estado"]
        ),
        (
            "lotacao_estimada",
            situacao["lotacao"]
        )
    }

REGRAS = [

    (
        "R1 origem",
        "∀l ∀e (usuario_esta_em(l) ∧ proximo_de(l,e) → origem(e))",
        r_origem
    ),

    (
        "R2 destino",
        "∀l ∀e (usuario_quer_ir(l) ∧ proximo_de(l,e) → destino(e))",
        r_destino
    ),

    (
        "R3 bloqueio",
        "∀e (fechada(e) → bloqueada(e))",
        r_bloqueio
    ),

    (
        "R4 acessibilidade",
        "∀e (precisa_acessibilidade ∧ elevador_em_manutencao(e) → inacessivel(e))",
        r_acessibilidade
    ),

    (
        "R5 alerta",
        "∀p ∀e (papel(p,e) ∧ inacessivel(e) → alerta(p,e))",
        r_alerta
    ),

    (
        "R6 integração",
        "∀e ∀l1 ∀l2 (pertence(e,l1) ∧ pertence(e,l2) ∧ l1 ≠ l2 → integracao(e))",
        r_integracao
    ),

    (
        "Bônus bloqueio (linha paralisada)",
        "∀e ((∃l pertence(e,l)) ∧ ∀l (pertence(e,l) → paralisada(l)) → bloqueada(e))",
        r_linha_paralisada
    ),

    (
        "R7 situação pelo horário",
        "horario_considerado(h) → situacao_metro ∧ lotacao_estimada",
        r_situacao_horario
    ),
]

def encadear_para_frente(
    fatos,
    regras,
    verbose=False
):
    """
    Aplica as regras repetidamente até que nenhum
    fato novo seja produzido.

    Isso gera um ponto fixo de inferência.
    """

    fatos = set(fatos)

    justificativas = {}

    rodada = 0

    while True:

        rodada += 1

        novos_na_rodada = set()

        for nome, _formula, regra in regras:

            fatos_produzidos = regra(fatos)

            for fato in fatos_produzidos - fatos:

                novos_na_rodada.add(fato)

                justificativas.setdefault(
                    fato,
                    nome
                )

        if verbose:
            print(
                f"Rodada {rodada}: "
                f"{len(novos_na_rodada)} fato(s) novo(s)"
            )

        if not novos_na_rodada:
            return fatos, justificativas

        fatos |= novos_na_rodada

_f, _j = encadear_para_frente(
    fatos_base(),
    REGRAS,
    verbose=True
)

print(
    "Integrações deduzidas pela R6:",
    sorted(
        e
        for (e,) in consultar(
            _f,
            "integracao"
        )
    )
)

Rodada 1: 5 fato(s) novo(s)
Rodada 2: 0 fato(s) novo(s)
Integrações deduzidas pela R6: ['Ana Rosa', 'Paraíso', 'Sé']


In [ ]:
# Célula 10 — o planejador (lógica + busca + baldeações)
TEMPO_POR_TRECHO = 2     # minutos por trecho
TEMPO_BALDEACAO = 5      # minutos extras por baldeação


def planejar(
    pedido,
    fechadas=(),
    manutencao=(),
    algoritmo="BFS",
    paralisadas=()
):
    """
    pedido = {
        "origem": (tipo, nome),
        "destino": (tipo, nome),
        "acessibilidade": bool,
        "horario": "HH:MM" ou None
    }

    tipo é "local" ou "estacao".
    """

    fatos = fatos_base()

    tipo_o, nome_o = pedido["origem"]
    tipo_d, nome_d = pedido["destino"]

    fatos.add(
        ("usuario_esta_em", nome_o)
        if tipo_o == "local"
        else ("usuario_esta_na_estacao", nome_o)
    )

    fatos.add(
        ("usuario_quer_ir", nome_d)
        if tipo_d == "local"
        else ("usuario_quer_ir_estacao", nome_d)
    )

    if pedido.get("acessibilidade", False):
        fatos.add(("precisa_acessibilidade", True))

    horario_foi_informado = bool(pedido.get("horario"))
    if horario_foi_informado:
        horario_utilizado = validar_horario(pedido["horario"])
        if horario_utilizado is None:
            raise ValueError(
                f"Horário inválido: {pedido['horario']!r} (use HH:MM, entre 00:00 e 23:59)"
            )
    else:
        horario_utilizado = agora_sp().strftime("%H:%M")

    fatos.add(("horario_considerado", horario_utilizado))

    for e in fechadas:
        fatos.add(("fechada", e))

    for e in manutencao:
        fatos.add(("elevador_em_manutencao", e))

    for l in paralisadas:
        fatos.add(("paralisada", l))

    fatos, justificativas = encadear_para_frente(
        fatos,
        REGRAS
    )

    origens = consultar(fatos, "origem")
    destinos = consultar(fatos, "destino")

    if not origens or not destinos:
        raise ValueError(
            f"Não foi possível deduzir origem/destino de "
            f"{pedido['origem']} → {pedido['destino']}"
        )

    origem = origens[0][0]
    destino = destinos[0][0]

    bloqueadas = {
        e
        for (e,) in consultar(
            fatos,
            "bloqueada"
        )
    }

    alertas = consultar(
        fatos,
        "alerta"
    )

    integracoes = sorted(
        e
        for (e,) in consultar(
            fatos,
            "integracao"
        )
    )

    linhas_paralisadas = sorted(
        l
        for (l,) in consultar(
            fatos,
            "paralisada"
        )
    )

    situacao_metro = next(
        (
            valor
            for (valor,) in consultar(
                fatos,
                "situacao_metro"
            )
        ),
        "desconhecida"
    )

    lotacao_estimada = next(
        (
            valor
            for (valor,) in consultar(
                fatos,
                "lotacao_estimada"
            )
        ),
        "desconhecida"
    )


    metro_fechado = situacao_metro in SITUACOES_SEM_OPERACAO

    if metro_fechado:
        caminho, visitados = None, []
        linhas_do_trecho = {}
    else:
        grafo, linhas_do_trecho = construir_grafo_ativo(
            LINHAS,
            linhas_paralisadas
        )

        buscar = bfs if algoritmo == "BFS" else dfs

        caminho, visitados = buscar(
            grafo,
            origem,
            destino,
            bloqueadas
        )

    if caminho:
        paradas = len(caminho) - 1

        n_bald, baldeacoes = contar_baldeacoes(
            caminho,
            linhas_do_trecho
        )

        segmentos = segmentos_do_caminho(
            caminho,
            linhas_do_trecho
        )

        tempo = (
            paradas * TEMPO_POR_TRECHO
            + n_bald * TEMPO_BALDEACAO
        )

    else:
        paradas = None
        n_bald = None
        tempo = None
        baldeacoes = []
        segmentos = []

    return {
        "origem": origem,
        "destino": destino,
        "algoritmo": algoritmo,

        "caminho": caminho,
        "visitados": visitados,

        "bloqueadas": sorted(bloqueadas),
        "linhas_paralisadas": linhas_paralisadas,

        "alertas": [
            f"{papel}: {e}"
            for papel, e in alertas
        ],

        "paradas": paradas,
        "n_baldeacoes": n_bald,
        "baldeacoes": baldeacoes,
        "segmentos": segmentos,

        "tempo_min": tempo,
        "integracoes": integracoes,

        "regras_usadas": sorted(
            set(justificativas.values())
        ),

        # R7
        "situacao_metro": situacao_metro,
        "lotacao_estimada": lotacao_estimada,
        "horario_utilizado": horario_utilizado,
        "horario_informado": horario_foi_informado,
        "metro_fechado": metro_fechado,
        "abertura": ABERTURA_METRO,
    }


def comparar_algoritmos(
    pedido,
    fechadas=(),
    manutencao=(),
    paralisadas=()
):
    """
    Roda BFS e DFS para a MESMA viagem
    e devolve {"BFS": resultado, "DFS": resultado}.
    """

    return {
        alg: planejar(
            pedido,
            fechadas,
            manutencao,
            alg,
            paralisadas
        )
        for alg in ("BFS", "DFS")
    }


def resumo_algoritmo(r):
    """
    Uma linha com paradas e esforço
    (estações visitadas) do algoritmo usado em r.
    """

    if r["caminho"] is None:
        return (
            f"{r['algoritmo']}: sem rota "
            f"({len(r['visitados'])} estações visitadas)"
        )

    return (
        f"{r['algoritmo']}: "
        f"{r['paradas']} paradas, "
        f"{len(r['visitados'])} estações visitadas"
    )


print(
    f"{'Viagem':<42}"
    f"{'BFS (paradas/visitadas)':>25}"
    f"{'DFS (paradas/visitadas)':>25}"
)

for o, d in [
    ("Tucuruvi", "Corinthians-Itaquera"),
    ("Vila Madalena", "Jabaquara"),
    ("Palmeiras-Barra Funda", "Vila Prudente"),
    ("Sé", "Japão-Liberdade")
]:
    comp = comparar_algoritmos({
        "origem": ("estacao", o),
        "destino": ("estacao", d),
        "horario": "12:00"              # horário fixo: a demonstração não depende da hora em que roda
    })

    b = comp["BFS"]
    dd = comp["DFS"]

    print(
        f"{o + ' → ' + d:<42}"
        f"{str(b['paradas']) + ' / ' + str(len(b['visitados'])):>25}"
        f"{str(dd['paradas']) + ' / ' + str(len(dd['visitados'])):>25}"
    )

Viagem                                      BFS (paradas/visitadas)  DFS (paradas/visitadas)
Tucuruvi → Corinthians-Itaquera                             22 / 52                  22 / 52
Vila Madalena → Jabaquara                                   14 / 37                  14 / 46
Palmeiras-Barra Funda → Vila Prudente                       16 / 49                  16 / 34
Sé → Japão-Liberdade                                          1 / 3                   1 / 12


In [ ]:
# Célula 11 — R4: intérprete (Llama) com validação, guardrails e modo offline


def normalizar(texto):
    """Minúsculas e sem acentos: 'São Bento' → 'sao bento'."""
    texto = unicodedata.normalize("NFD", texto.lower())
    return "".join(
        c for c in texto
        if unicodedata.category(c) != "Mn"
    )


# Apelidos → nome oficial (estação ou local). Assim "jogo do Palmeiras" vira Nubank Parque
# (a estação mais próxima é a Palmeiras-Barra Funda) e nunca a Itaquera.
APELIDOS = {
    # Palmeiras
    "jogo do Palmeiras": "Nubank Parque",
    "estádio do Palmeiras": "Nubank Parque",
    "Allianz Parque": "Nubank Parque",
    "Palmeiras": "Nubank Parque",
    # Corinthians
    "jogo do Corinthians": "Neo Química Arena",
    "estádio do Corinthians": "Neo Química Arena",
    "Arena Corinthians": "Neo Química Arena",
    "Itaquerão": "Neo Química Arena",
    "Corinthians": "Neo Química Arena",
    # Estação pelo nome curto
    "Barra Funda": "Palmeiras-Barra Funda",
}
assert all(v in ESTACOES or v in LOCAIS for v in APELIDOS.values()), "apelido apontando para nome inexistente"
LOCAIS_DE_TORCIDA = {"Nubank Parque", "Neo Química Arena"}


def resolver_nome(nome):
    """GUARDRAIL: só aceita nomes que existem de verdade (52 estações + locais). Senão, None."""
    if not nome or not isinstance(nome, str):
        return None

    alvo = normalizar(nome).strip()

    for estacao in ESTACOES:
        if normalizar(estacao) == alvo:
            return ("estacao", estacao)

    for local in LOCAIS:
        if normalizar(local) == alvo:
            return ("local", local)

    for apelido, oficial in APELIDOS.items():
        if normalizar(apelido) == alvo:
            return resolver_nome(oficial)

    return None


PROMPT_INTERPRETE = """Você é o módulo de INTERPRETAÇÃO do MetrôBot SP.

Sua única tarefa é transformar o pedido do passageiro em JSON.

Estações válidas: {estacoes}

Locais válidos: {locais}

Responda APENAS com um JSON neste formato:

{{
  "origem": "<nome exato de estação ou local, ou null>",
  "destino": "<nome exato de estação ou local, ou null>",
  "acessibilidade": <true ou false>,
  "horario": "<HH:MM ou null>"
}}

Regras:

- Use SOMENTE nomes das listas acima.
- "jogo do Palmeiras", "estádio do Palmeiras" ou "Allianz Parque" significam "Nubank Parque".
- "jogo do Corinthians", "Itaquerão" ou "Arena Corinthians" significam "Neo Química Arena".
- NUNCA troque um time pelo outro: Palmeiras é Nubank Parque, Corinthians é Neo Química Arena.
- "acessibilidade" é true se o passageiro mencionar cadeira de rodas,
  mobilidade reduzida, muletas, carrinho de bebê ou precisar de elevador.
- Se o passageiro informar um horário, converta para o formato HH:MM.
- Exemplos:
    "às 18:30" → "18:30"
    "às 18h30" → "18:30"
    "às 8 da manhã" → "08:00"
    "às 7 da noite" → "19:00"
- Se o passageiro não informar horário, use null.
- Nunca invente um horário.
- Se não souber algum campo, use null.
"""


def extrair_horario(texto):
    """
    Tenta identificar um horário informado pelo usuário.

    Aceita: 18:30 | 18h30 | 18h | 8 da manhã | 3 da tarde | 7 da noite | 8h da noite
    Retorna "HH:MM" ou None.
    """

    t = normalizar(texto)

    m = re.search(
        r"\b(\d{1,2})"                                          # hora
        r"(?:[:h]([0-5]\d)|h\b|(?=\s+da\s+(?:manha|tarde|noite|madrugada)\b))"   # :30 | h30 | h | (da manhã…)
        r"(?:\s+da\s+(manha|tarde|noite|madrugada)\b)?",         # período do dia (opcional)
        t
    )

    if not m:
        return None

    hora = int(m.group(1))
    minuto = int(m.group(2) or 0)
    periodo = m.group(3)

    # "7 da noite" → 19:00 | "3 da tarde" → 15:00 | "12 da noite" → 00:00
    if periodo in ("tarde", "noite") and 1 <= hora <= 11:
        hora += 12
    elif periodo in ("noite", "madrugada") and hora == 12:
        hora = 0

    return validar_horario(f"{hora}:{minuto:02d}")


def interpretar_offline(texto):
    """Plano B sem LLM: procura nomes conhecidos no texto, na ordem em que aparecem."""

    texto_min = texto.lower()
    texto_sem = normalizar(texto)

    candidatos = (
        [(n, "estacao") for n in ESTACOES]
        + [(n, "local") for n in LOCAIS]
        + [(n, "apelido") for n in APELIDOS]
    )

    candidatos.sort(
        key=lambda c: len(c[0]),
        reverse=True
    )

    ocupado = [False] * len(texto_min)
    encontrados = []

    for nome, tipo in candidatos:
        buscas = [
            (texto_min, nome.lower())
        ]

        if len(nome) > 4 and len(texto_sem) == len(texto_min):
            buscas.append(
                (
                    texto_sem,
                    normalizar(nome)
                )
            )

        for base, padrao in buscas:

            for m in re.finditer(
                r"(?<!\w)" + re.escape(padrao) + r"(?!\w)",
                base
            ):

                if not any(
                    ocupado[m.start():m.end()]
                ):
                    encontrados.append(
                        (
                            m.start(),
                            APELIDOS.get(nome, nome)      # apelido → nome oficial
                        )
                    )

                    for i in range(
                        m.start(),
                        m.end()
                    ):
                        ocupado[i] = True

    encontrados.sort()

    palavras_acess = [
        "cadeira de rodas",
        "acessibilidade",
        "mobilidade",
        "muleta",
        "carrinho de bebe",
        "elevador"
    ]

    return {
        "origem": (
            encontrados[0][1]
            if len(encontrados) > 0
            else None
        ),
        "destino": (
            encontrados[1][1]
            if len(encontrados) > 1
            else None
        ),
        "acessibilidade": any(
            p in texto_sem
            for p in palavras_acess
        ),
        "horario": extrair_horario(texto),
    }


def interpretar_pedido(texto):
    """Texto livre → pedido validado. Usa o Llama; se falhar, cai no modo offline."""

    if PROVEDOR == "offline":
        bruto = interpretar_offline(texto)
        fonte = "offline"

    else:
        sistema = PROMPT_INTERPRETE.format(
            estacoes=", ".join(ESTACOES),
            locais=", ".join(LOCAIS)
        )

        try:
            resposta = chamar_llm(
                [
                    {
                        "role": "system",
                        "content": sistema
                    },
                    {
                        "role": "user",
                        "content": texto
                    }
                ],
                modo_json=True
            )

            bruto = json.loads(resposta)

            if not isinstance(bruto, dict):
                raise ValueError(
                    "o LLM não devolveu um objeto JSON"
                )

            fonte = PROVEDOR

        except Exception as erro:
            print(
                f"⚠️ LLM indisponível "
                f"({type(erro).__name__}). "
                "Usando modo offline."
            )

            bruto = interpretar_offline(texto)
            fonte = "offline"

    origem = resolver_nome(
        bruto.get("origem")
    )

    destino = resolver_nome(
        bruto.get("destino")
    )

    # GUARDRAIL DE TORCIDA: se o texto cita o Palmeiras/Corinthians (ou seus estádios) e o LLM
    # devolveu outro lugar, o algoritmo decide e usa a leitura por nomes conhecidos.
    if fonte != "offline":
        offline = interpretar_offline(texto)
        citados = {offline["origem"], offline["destino"]} & LOCAIS_DE_TORCIDA
        devolvidos = {p[1] for p in (origem, destino) if p}

        if not citados <= devolvidos:
            print(
                "⚠️ O LLM ignorou "
                + ", ".join(sorted(citados - devolvidos))
                + ". Usando a leitura por nomes conhecidos."
            )
            bruto = offline
            fonte = f"{fonte} + correção offline"
            origem = resolver_nome(bruto.get("origem"))
            destino = resolver_nome(bruto.get("destino"))

    if origem is None or destino is None:
        return (
            None,
            f"Não entendi origem/destino "
            f"(resposta bruta: {bruto})"
        )

    # GUARDRAIL: o LLM pode devolver "false" (texto) ou um horário malformado.
    acess = bruto.get("acessibilidade")
    if isinstance(acess, str):
        acess = acess.strip().lower() in ("true", "sim", "1")

    pedido = {
        "origem": origem,
        "destino": destino,
        "acessibilidade": bool(acess),
        "horario": validar_horario(bruto.get("horario"))
    }

    return (
        pedido,
        f"Interpretado via {fonte}"
    )


for texto in [
    "Estou na Catedral da Sé e quero ir ao Terminal Rodoviário Jabaquara",
    "Preciso ir do MASP até a Neo Química Arena, estou de cadeira de rodas",
    "Quero ir da Sé até a Avenida Paulista"
]:
    bruto = interpretar_offline(texto)

    print(
        texto,
        "→",
        resolver_nome(bruto["origem"]),
        resolver_nome(bruto["destino"]),
        bruto["acessibilidade"]
    )

Estou na Catedral da Sé e quero ir ao Terminal Rodoviário Jabaquara → ('local', 'Catedral da Sé') ('local', 'Terminal Rodoviário Jabaquara') False
Preciso ir do MASP até a Neo Química Arena, estou de cadeira de rodas → ('local', 'MASP') ('local', 'Neo Química Arena') True
Quero ir da Sé até a Avenida Paulista → ('estacao', 'Sé') None False


In [ ]:
# Célula 12 — R4: narrador + R7: situação do metrô pelo horário


def narrar_offline(r):
    """Narrador sem LLM: monta frases somente com os dados do planejador."""

    # Metrô fechado: nada de rota — só explica e diz quando volta a operar
    if r.get("metro_fechado"):
        return (
            f"🕐 O metrô está fechado às {r['horario_utilizado']}, então não há rota de "
            f"{r['origem']} até {r['destino']} neste horário. "
            f"A operação começa às {r.get('abertura', ABERTURA_METRO)}."
        )

    # Caso não exista rota
    if r["caminho"] is None:
        motivos = []

        if r["bloqueadas"]:
            motivos.append(
                "estações bloqueadas: " + ", ".join(r["bloqueadas"])
            )

        if r["linhas_paralisadas"]:
            motivos.append(
                "linhas paralisadas: " + ", ".join(r["linhas_paralisadas"])
            )

        detalhe = (
            "; ".join(motivos)
            if motivos
            else "não há ligação entre os dois pontos"
        )

        texto = (
            f"Não existe rota de {r['origem']} até {r['destino']} "
            f"({detalhe})."
        )

    # Caso origem e destino sejam a mesma estação
    elif r["paradas"] == 0:
        texto = (
            f"Você já está em {r['origem']}: "
            "não é preciso embarcar."
        )

    else:
        partes = []

        # Descrição dos trechos da viagem
        for i, (linha, ini, fim, n) in enumerate(r["segmentos"]):

            if i == 0:
                partes.append(
                    f"Embarque em {ini} na {linha} "
                    f"e siga {n} parada(s) até {fim}."
                )

            else:
                partes.append(
                    f"Na estação {ini}, troque para a {linha} "
                    f"e siga {n} parada(s) até {fim}."
                )

        partes.append(
            f"Total: {r['paradas']} parada(s), "
            f"{r['n_baldeacoes']} baldeação(ões), "
            f"cerca de {r['tempo_min']} minutos."
        )

        # Alertas de acessibilidade/manutenção
        if r["alertas"]:
            partes.append(
                "Atenção: "
                + "; ".join(r["alertas"])
                + " (elevador em manutenção)."
            )

        texto = " ".join(partes)

    situacao = r.get("situacao_metro", "desconhecida")
    lotacao = r.get("lotacao_estimada", "desconhecida")

    if situacao == "fechado":
        texto += (
            "\n🕐 R7: O metrô está fechado neste momento."
        )

    elif situacao == "prestes a abrir":
        texto += (
            "\n🕐 R7: O metrô está prestes a abrir, "
            "com movimento muito baixo."
        )

    elif situacao == "acabou de abrir":
        texto += (
            "\n🕐 R7: O metrô acabou de abrir e "
            "o movimento tende a estar baixo."
        )

    elif situacao == "horário de pico":
        texto += (
            "\n🕐 R7: É horário de pico e o metrô está "
            f"{lotacao}."
        )

    elif situacao == "horário intermediário":
        texto += (
            "\n🕐 R7: O metrô está em um período de "
            f"{lotacao}."
        )

    elif situacao == "horário noturno":
        texto += (
            "\n🕐 R7: O movimento tende a ser moderado "
            "neste horário."
        )

    elif situacao == "prestes a fechar":
        texto += (
            "\n🕐 R7: O metrô está se aproximando "
            "do fim da operação, com "
            f"{lotacao}."
        )

    else:
        texto += (
            f"\n🕐 R7: Situação estimada: {situacao}. "
            f"Movimento: {lotacao}."
        )

    return texto

PROMPT_NARRADOR = """Você é o NARRADOR do MetrôBot SP.

Explique a rota ao passageiro em português,
em no máximo 6 frases curtas e simpáticas.

Use SOMENTE os dados do JSON.

Não invente horários, linhas, estações ou atrações.

Descreva a viagem trecho a trecho usando "trechos".

Ao trocar de linha, use:
"Na estação X, troque para a Linha Y".

Se "metro_fechado" for true, diga que o metrô está fechado no "horario_utilizado",
que por isso não há rota agora e que ele volta a operar no horário de "abertura".
Não descreva trajeto nem baldeações nesse caso.

Se "caminho" for null, explique que não há rota
e cite as estações bloqueadas e/ou linhas paralisadas.

Se houver "alertas", destaque-os.

A situação do metrô é uma ESTIMATIVA baseada no horário.
Não diga que a lotação foi medida em tempo real.

Use "horario_utilizado" como o horário da viagem.
Se "horario_foi_informado" for false, esse horário é o momento atual
(horário de Brasília).

Informe de maneira natural se o metrô pode estar:
- cheio;
- com movimento moderado;
- com movimento baixo;
- recém-aberto;
- prestes a abrir;
- prestes a fechar;
- fechado.

Responda em no máximo 6 frases curtas.
"""

def _cita(texto, estacao):
    """Verifica se uma estação aparece no texto."""
    return re.search(
        r"(?<!\w)" + re.escape(estacao) + r"(?!\w)",
        texto
    ) is not None


def motivo_alucinacao(texto, r):
    """
    Devolve o MOTIVO pelo qual o texto do LLM deve ser descartado,
    ou None se ele passou em todos os guardrails.

    1. Não pode citar uma estação que não faça parte da situação.
    2. Números usados precisam estar presentes nos dados.
    3. Todas as baldeações precisam ser mencionadas.
    """

    permitidas = (
        set(r["caminho"] or [])
        | {r["origem"], r["destino"]}
        | set(r["bloqueadas"])
    )

    for estacao in ESTACOES:
        if estacao not in permitidas and _cita(texto, estacao):
            return f"citou estação fora da situação: {estacao}"

    permitidos = {0, 1, 2, 3, r["n_baldeacoes"], r["tempo_min"]}
    permitidos |= {n for *_, n in r["segmentos"]}

    permitidos |= {int(n) for n in re.findall(r"\d+", r.get("horario_utilizado") or "")}
    permitidos |= {int(n) for n in re.findall(r"\d+", r.get("abertura") or "")}

    if len(r["segmentos"]) <= 1 or re.search(r"total|ao todo|no todo", texto.lower()):
        permitidos.add(r["paradas"])

    for numero in re.findall(r"\d+", texto):
        if int(numero) not in permitidos:
            return f"número que não está nos dados: {numero}"

    for estacao, _linha in r["baldeacoes"] or []:
        if not _cita(texto, estacao):
            return f"não mencionou a baldeação em {estacao}"

    return None


def narrador_alucinou(texto, r):
    """True se o texto do LLM deve ser descartado (veja o motivo em motivo_alucinacao)."""
    return motivo_alucinacao(texto, r) is not None

def montar_dados_narrador(resultado):
    """Dados (e SÓ eles) que o LLM pode usar para narrar a rota."""

    return {
        "origem": resultado["origem"],
        "destino": resultado["destino"],
        "caminho": resultado["caminho"],
        "paradas_total": resultado["paradas"],
        "baldeacoes": resultado["n_baldeacoes"],
        "trechos": [
            {"linha": linha, "de": inicio, "ate": fim, "paradas": paradas}
            for linha, inicio, fim, paradas in resultado["segmentos"]
        ],
        "onde_trocar": [
            {"estacao": estacao, "para_linha": linha}
            for estacao, linha in resultado["baldeacoes"]
        ],
        "tempo_min": resultado["tempo_min"],
        "bloqueadas": resultado["bloqueadas"],
        "linhas_paralisadas": resultado["linhas_paralisadas"],
        "alertas": resultado["alertas"],

        "situacao_metro": resultado.get("situacao_metro", "desconhecida"),
        "lotacao_estimada": resultado.get("lotacao_estimada", "desconhecida"),
        "horario_utilizado": resultado["horario_utilizado"],
        "horario_foi_informado": resultado.get("horario_informado", False),
        "metro_fechado": resultado.get("metro_fechado", False),
        "abertura": resultado.get("abertura", ABERTURA_METRO),
    }


def narrar(resultado):
    """
    Transforma o resultado do planejador em uma explicação amigável.

    Com LLM: usa o modelo configurado. Sem LLM: narrar_offline().
    O guardrail descarta respostas do LLM com informações que não estão nos dados.
    """

    if PROVEDOR == "offline":
        return narrar_offline(resultado)

    dados = montar_dados_narrador(resultado)

    try:
        texto = chamar_llm(
            [
                {"role": "system", "content": PROMPT_NARRADOR},
                {"role": "user", "content": json.dumps(dados, ensure_ascii=False)}
            ]
        )

    except Exception as erro:
        return (
            narrar_offline(resultado)
            + f"  (narrador offline: {type(erro).__name__})"
        )

    if not texto:
        return narrar_offline(resultado) + "  (texto do LLM vazio)"

    motivo = motivo_alucinacao(texto, resultado)

    if motivo:
        return (
            narrar_offline(resultado)
            + f"  (texto do LLM descartado pelo guardrail: {motivo})"
        )

    return texto


# ---------------------------------------------------------------------------
# Easter egg do grupo (torcedor palmeirense 💚🐷)
# ---------------------------------------------------------------------------

EASTER_EGGS_TORCEDOR = {
    "Corinthians-Itaquera":
        "😏🐷 Rumo ao Itaquerão torcer contra o Timão? "
        "Boa viagem — e Avanti, Palestra! 💚",

    "Neo Química Arena":
        "😏🐷 Rumo ao Itaquerão torcer contra o Timão? "
        "Boa viagem — e Avanti, Palestra! 💚",

    "Nubank Parque":
        "💚🐷 Rumo à casa do Verdão! "
        "Avanti, Palestra! 🏆",
}


def mensagem_torcedor(pedido):
    """
    Easter egg pessoal do grupo — aparece SOMENTE quando o DESTINO é um dos
    lugares de EASTER_EGGS_TORCEDOR (sair de lá não conta).

    Fique à vontade para editar ou remover as mensagens.
    """

    _tipo, nome_destino = pedido["destino"]

    return EASTER_EGGS_TORCEDOR.get(nome_destino)

print(
    narrar_offline(
        planejar(
            {
                "origem": ("estacao", "Tucuruvi"),
                "destino": ("estacao", "Corinthians-Itaquera"),
                "horario": "12:00"
            }
        )
    )
)

Embarque em Tucuruvi na Linha 1-Azul e siga 10 parada(s) até Sé. Na estação Sé, troque para a Linha 3-Vermelha e siga 12 parada(s) até Corinthians-Itaquera. Total: 22 parada(s), 1 baldeação(ões), cerca de 49 minutos.
🕐 R7: O metrô está em um período de movimento moderado.


In [ ]:
# Célula 12b — diagnóstico do guardrail (opcional): por que o texto do LLM foi rejeitado?
def diagnosticar_guardrail(pedido=None):
    """Roda o planejador, pede o texto ao LLM e mostra o veredito do guardrail."""

    if PROVEDOR == "offline":
        print("Modo offline: não há texto de LLM para diagnosticar. "
              "Troque PROVEDOR na célula 2 para testar o guardrail com o Llama.")
        return

    pedido = pedido or {
        "origem": ("estacao", "Tucuruvi"),
        "destino": ("estacao", "Corinthians-Itaquera"),
        "horario": "18:30",
    }

    resultado = planejar(pedido)
    dados = montar_dados_narrador(resultado)

    try:
        texto_llm = chamar_llm(
            [
                {"role": "system", "content": PROMPT_NARRADOR},
                {"role": "user", "content": json.dumps(dados, ensure_ascii=False)},
            ]
        )
    except Exception as erro:
        print(f"⚠️ LLM indisponível ({type(erro).__name__}): {erro}")
        return

    print("=== TEXTO GERADO PELO LLM ===")
    print(texto_llm)

    motivo = motivo_alucinacao(texto_llm, resultado)
    print("\n=== RESULTADO DO GUARDRAIL ===")
    print("Alucinou?", motivo is not None)
    if motivo:
        print("Motivo:", motivo)


diagnosticar_guardrail()

=== TEXTO GERADO PELO LLM ===
Você embarca na Linha 1‑Azul em Tucuruvi e segue até a estação Sé, passando por 10 paradas.  
Na estação Sé, troque para a Linha 3‑Vermelha.  
Continue na Linha 3‑Vermelha até chegar em Corinthians‑Itaquera, percorrendo mais 12 paradas.  
A viagem tem cerca de 49 minutos e 22 paradas no total.  
O metrô está em horário de pico e provavelmente cheio às 18:30.  
Não há alertas nem bloqueios, então a rota está disponível.

=== RESULTADO DO GUARDRAIL ===
Alucinou? False


In [ ]:
# Célula 13 — R5: desenho das 3 linhas em HTML (com as CORES)
def desenhar_linhas(r):
    """Uma coluna por linha, na cor oficial do modelo. Marca rota, origem/destino, visitadas e bloqueadas."""
    caminho = set(r["caminho"] or [])
    visitados = set(r["visitados"])
    bloqueadas = set(r["bloqueadas"])
    integracoes = set(r["integracoes"])
    paralisadas = set(r["linhas_paralisadas"])
    troca_em = dict(r["baldeacoes"] or [])

    usadas = {}
    for linha, ini, fim, _n in r["segmentos"]:
        i0, i1 = r["caminho"].index(ini), r["caminho"].index(fim)
        for e in r["caminho"][i0:i1 + 1]:
            usadas.setdefault(e, set()).add(linha)

    def ponto(cor_fundo, cor_borda):
        return (f"<span style='display:inline-block;width:12px;height:12px;border-radius:50%;"
                f"background:{cor_fundo};border:3px solid {cor_borda};flex:none'></span>")

    colunas = []
    for nome_linha, estacoes in LINHAS.items():
        cor = CORES[nome_linha]
        titulo = nome_linha + (" — ⛔ paralisada" if nome_linha in paralisadas else "")
        itens = []
        for e in estacoes:
            fundo_item, negrito = "transparent", "normal"
            if e in bloqueadas:
                dot, marca, fundo_item = ponto("#ffffff", "#d32f2f"), "⛔ bloqueada", "#ffebee"
            elif e == r["origem"]:
                dot, marca, negrito = ponto("#0d47a1", "#0d47a1"), "⭐ origem", "bold"
            elif e == r["destino"]:
                dot, marca, negrito = ponto("#0d47a1", "#0d47a1"), "⭐ destino", "bold"
            elif nome_linha in usadas.get(e, ()):
                dot, marca, negrito = ponto(cor, cor), "rota", "bold"
            elif e in visitados:
                dot, marca = ponto("#9e9e9e", "#9e9e9e"), "visitada pela busca"
            else:
                dot, marca = ponto("#ffffff", "#bdbdbd"), ""
            extras = ""
            if e in integracoes:
                extras += " ⇄"
            if e in troca_em and nome_linha == troca_em[e]:
                extras += f" 🔁 troque para a {troca_em[e]}"
            itens.append(
                f"<div style='display:flex;align-items:center;gap:6px;padding:1px 4px;font-size:12px;"
                f"background:{fundo_item};font-weight:{negrito}'>{dot}"
                f"<span style='min-width:135px'>{e}{extras}</span>"
                f"<span style='color:#666;font-weight:normal'>{marca}</span></div>")
        colunas.append(
            f"<div style='border-left:5px solid {cor};padding-left:6px;margin:4px 10px 4px 0'>"
            f"<div style='font-weight:bold;color:{cor};margin-bottom:3px'>{titulo}</div>"
            + "".join(itens) + "</div>")
    legenda = ("<div style='font-size:11px;color:#555;margin-bottom:4px'>⭐ origem/destino · "
               "bolinha na cor da linha = rota · cinza = visitada pela busca · "
               "⛔ bloqueada · ⇄ integração (deduzida pela R6) · 🔁 baldeação</div>")
    return ("<div style='background:#fff;color:#222;padding:8px;border-radius:8px;font-family:sans-serif'>"
            + legenda + "<div style='display:flex;flex-wrap:wrap'>" + "".join(colunas) + "</div></div>")

In [ ]:
# Célula 14 — R5: interface com ipywidgets
def montar_painel():
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output

    estilo = {"description_width": "initial"}
    opcoes = ([(f"📍 {local}", ("local", local)) for local in LOCAIS] +
              [(f"🚇 {estacao}", ("estacao", estacao)) for estacao in ESTACOES])

    txt_pedido = widgets.Textarea(
        placeholder="Ex.: Estou no MASP e quero ir à Neo Química Arena, preciso de acessibilidade",
        layout=widgets.Layout(width="95%", height="60px"))
    btn_interpretar = widgets.Button(description="🤖 Interpretar pedido", button_style="info")
    dd_origem = widgets.Dropdown(options=opcoes, value=("estacao", "Vila Madalena"),
                                 description="Origem:", style=estilo)
    dd_destino = widgets.Dropdown(options=opcoes, value=("local", "Terminal Rodoviário Jabaquara"),
                                  description="Destino:", style=estilo)
    chk_acess = widgets.Checkbox(description="Preciso de acessibilidade")
    txt_horario = widgets.Text(description="Horário:", placeholder="HH:MM (vazio = agora)",
                               layout=widgets.Layout(width="230px"))
    rb_algoritmo = widgets.RadioButtons(options=["BFS", "DFS"], description="Busca:")
    sel_fechadas = widgets.SelectMultiple(options=ESTACOES, description="Fechadas:", rows=6, style=estilo)
    sel_manut = widgets.SelectMultiple(options=ESTACOES, description="Elevador 🛠️:", rows=6, style=estilo)
    sel_paralisadas = widgets.SelectMultiple(options=list(LINHAS), description="Linha paralisada:",
                                             rows=3, style=estilo)
    btn_buscar = widgets.Button(description="🚇 Buscar rota", button_style="success")
    saida = widgets.Output()

    def ao_interpretar(_):
        with saida:
            clear_output()
            pedido, msg = interpretar_pedido(txt_pedido.value)
            print(msg)
            if pedido:
                dd_origem.value = pedido["origem"]
                dd_destino.value = pedido["destino"]
                chk_acess.value = pedido["acessibilidade"]
                txt_horario.value = pedido["horario"] or ""
                print("✅ Campos preenchidos. Confira e clique em 'Buscar rota'.")

    def ao_buscar(_):
        with saida:
            clear_output()
            horario = validar_horario(txt_horario.value) if txt_horario.value.strip() else None
            if txt_horario.value.strip() and horario is None:
                print("⚠️ Horário inválido. Use HH:MM (ex.: 18:30) ou deixe em branco para usar o horário atual.")
                return
            pedido = {"origem": dd_origem.value, "destino": dd_destino.value,
                      "acessibilidade": chk_acess.value, "horario": horario}
            try:
                comp = comparar_algoritmos(pedido, sel_fechadas.value, sel_manut.value, sel_paralisadas.value)
            except Exception as erro:
                print(f"⚠️ Não consegui planejar a viagem: {erro}")
                return
            r = comp[rb_algoritmo.value]
            display(HTML(f"<h4>{r['algoritmo']}: {r['origem']} → {r['destino']}</h4>"))
            mensagem = mensagem_torcedor(pedido)
            if mensagem:
                print(mensagem)
            print("🗣️", narrar(r))
            if r["metro_fechado"]:          # horário validado primeiro: fechado = sem rota, sem desenho
                print(f"📜 Regras disparadas: {', '.join(r['regras_usadas'])}")
                return
            if r["caminho"]:
                print(f"🚏 Paradas: {r['paradas']}   🔁 Baldeações: {r['n_baldeacoes']}   "
                      f"⏱️ Tempo estimado: {r['tempo_min']} min")
                print("🧭 Trechos: " + " | ".join(f"{l}: {i} → {f} ({n} paradas)" for l, i, f, n in r["segmentos"]))
                if r["baldeacoes"]:
                    print("📍 Onde trocar: " + " | ".join(f"{e} → {l}" for e, l in r["baldeacoes"]))
            print("🔎 Esforço —", resumo_algoritmo(comp["BFS"]), "|", resumo_algoritmo(comp["DFS"]))
            print(f"📜 Regras disparadas: {', '.join(r['regras_usadas'])}")
            display(HTML(desenhar_linhas(r)))

    btn_interpretar.on_click(ao_interpretar)
    btn_buscar.on_click(ao_buscar)

    return widgets.VBox([
        widgets.HTML("<h3>🚇 MetrôBot SP 2.0 — Linhas 1, 2 e 3</h3>"),
        txt_pedido, btn_interpretar,
        widgets.HBox([dd_origem, dd_destino]),
        widgets.HBox([chk_acess, txt_horario, rb_algoritmo]),
        widgets.HBox([sel_fechadas, sel_manut, sel_paralisadas]),
        btn_buscar, saida,
    ])

In [ ]:
# Célula 15 — testes automatizados
# 6 casos obrigatórios + 2 casos
# + testes complementares

def _rodar_testes():
    """Roda os casos obrigatórios e os casos extras do grupo,
    imprimindo o resultado OBTIDO x ESPERADO de cada um.
    """

    global RELOGIO_FIXO
    RELOGIO_FIXO = "12:00"      # sem horário informado, o "agora" vale 12:00 → testes não dependem da hora real

    def viagem(o, d, **cenario):
        return planejar(
            {
                "origem": ("estacao", o),
                "destino": ("estacao", d)
            },
            **cenario
        )

    resultados = []

    def caso(numero, titulo, obtido, esperado):
        ok = obtido == esperado
        resultados.append(ok)

        marca = "✅" if ok else "❌"

        print(f"{marca} Caso {numero} — {titulo}")
        print(f"    obtido:   {obtido}")

        if not ok:
            print(f"    esperado: {esperado}")


    print("🔧 Sanity check (modelagem): ", end="")

    assert len(GRAFO) == 52 and len(ESTACOES) == 52

    assert all(
        len(v) == len(set(v))
        for v in GRAFO.values()
    ), "vizinhos duplicados"

    assert LINHAS_DO_TRECHO[
        ("Paraíso", "Ana Rosa")
    ] == {
        "Linha 1-Azul",
        "Linha 2-Verde"
    }

    print(
        "52 estações, sem vizinhos duplicados, "
        "Paraíso–Ana Rosa nas 2 linhas ✅\n"
    )

    # ==========================================================
    # 6 CASOS OBRIGATÓRIOS
    # ==========================================================

    print("--- 6 casos obrigatórios do enunciado (BFS) ---")

    # Caso 1
    r = viagem(
        "Tucuruvi",
        "Corinthians-Itaquera"
    )

    caso(
        1,
        "Tucuruvi → Corinthians-Itaquera",
        (
            r["paradas"],
            r["n_baldeacoes"],
            r["baldeacoes"]
        ),
        (
            22,
            1,
            [("Sé", "Linha 3-Vermelha")]
        )
    )

    # Caso 2
    r = viagem(
        "Vila Madalena",
        "Jabaquara"
    )

    caso(
        2,
        "Vila Madalena → Jabaquara",
        (
            r["paradas"],
            r["n_baldeacoes"],
            r["baldeacoes"]
        ),
        (
            14,
            1,
            [("Ana Rosa", "Linha 1-Azul")]
        )
    )

    # Caso 3
    r = viagem(
        "Palmeiras-Barra Funda",
        "Vila Prudente"
    )

    caso(
        3,
        "Palmeiras-Barra Funda → Vila Prudente",
        (
            r["paradas"],
            r["n_baldeacoes"],
            r["baldeacoes"]
        ),
        (
            16,
            2,
            [
                ("Sé", "Linha 1-Azul"),
                ("Ana Rosa", "Linha 2-Verde")
            ]
        )
    )

    # Caso 4
    r = viagem(
        "Tucuruvi",
        "Brás",
        fechadas=["Sé"]
    )

    caso(
        4,
        "Tucuruvi → Brás, com a Sé fechada",
        r["caminho"],
        None
    )

    # Caso 5
    r = viagem(
        "Vila Madalena",
        "Jabaquara",
        fechadas=["Paraíso"]
    )

    caso(
        5,
        "Vila Madalena → Jabaquara, com o Paraíso fechado",
        r["caminho"],
        None
    )

    # Caso 6
    r = viagem(
        "Vila Prudente",
        "Jabaquara",
        fechadas=["Paraíso"]
    )

    caso(
        6,
        "Vila Prudente → Jabaquara, com o Paraíso fechado "
        "(desvio por Ana Rosa)",
        (
            r["paradas"],
            "Ana Rosa" in r["caminho"],
            "Paraíso" in r["caminho"]
        ),
        (
            13,
            True,
            False
        )
    )

    # ==========================================================
    # CASOS EXTRAS
    # ==========================================================

    print("\n--- Casos Extras ---")

    r = viagem(
        "Sé",
        "Luz"
    )

    caso(
        7,
        "R6 — integrações são DEDUZIDAS pelo motor "
        "(ninguém digitou à mão)",
        sorted(r["integracoes"]),
        sorted([
            "Sé",
            "Paraíso",
            "Ana Rosa"
        ])
    )

    # ----------------------------------------------------------
    # Caso 8
    # ----------------------------------------------------------

    r = planejar(
        {
            "origem": ("estacao", "Tucuruvi"),
            "destino": ("estacao", "Jabaquara"),
            "horario": "08:00"
        },

    )

    caso(
        8,
        "R7 — horário informado: 08:00",
        (
            r.get("situacao_metro"),
            r.get("lotacao_estimada"),
            r.get("horario_utilizado")
        ),
        (
            "horário de pico",
            "provavelmente cheio",
            "08:00"
        )
    )

    print(
        f"\n{sum(resultados)}/{len(resultados)} casos passaram."
    )

    assert all(
        resultados
    ), "Há casos falhando — veja ❌ acima."

    # ==========================================================
    # TESTES COMPLEMENTARES
    # ==========================================================

    print(
        "\n--- Testes complementares "
        "(busca, lógica, intérprete, narrador, R7, easter egg) ---"
    )

    # ----------------------------------------------------------
    # DFS
    # ----------------------------------------------------------

    assert (
        viagem(
            "Tucuruvi",
            "Corinthians-Itaquera",
            algoritmo="DFS"
        )["paradas"] == 22
    )

    # ----------------------------------------------------------
    # Comparação BFS x DFS
    # ----------------------------------------------------------

    b, d = comparar_algoritmos(
        {
            "origem": ("estacao", "Sé"),
            "destino": ("estacao", "Japão-Liberdade")
        }
    ).values()

    assert (
        b["paradas"] == d["paradas"] == 1
        and len(d["visitados"]) > len(b["visitados"])
    )

    # ----------------------------------------------------------
    # R2 — locais próximos às estações
    # ----------------------------------------------------------

    r = planejar(
        {
            "origem": ("local", "Catedral da Sé"),
            "destino": ("local", "Pinacoteca")
        }
    )

    assert r["destino"] == "Luz"
    assert r["paradas"] == 2

    # ----------------------------------------------------------
    # R4 + R5 — acessibilidade/manutenção
    # ----------------------------------------------------------

    r = planejar(
        {
            "origem": ("estacao", "Sé"),
            "destino": ("estacao", "Luz"),
            "acessibilidade": True
        },
        manutencao=["Luz"]
    )
    assert "destino: Luz" in r["alertas"]

    # ----------------------------------------------------------
    # Passar por estação ≠ desembarcar nela
    # ----------------------------------------------------------

    r = planejar(
        {
            "origem": ("estacao", "Sé"),
            "destino": ("estacao", "Tiradentes"),
            "acessibilidade": True
        },
        manutencao=["Luz"]
    )

    assert r["alertas"] == []
    assert "Luz" in r["caminho"]

    # ----------------------------------------------------------
    # Paralisação de linha
    # ----------------------------------------------------------

    assert (
        viagem(
            "Tucuruvi",
            "Brás",
            paralisadas=["Linha 3-Vermelha"]
        )["caminho"] is None
    )

    assert (
        viagem(
            "Tucuruvi",
            "Jabaquara",
            paralisadas=["Linha 3-Vermelha"]
        )["paradas"] == 22
    )

    assert (
        viagem(
            "Paraíso",
            "Jabaquara",
            paralisadas=["Linha 2-Verde"]
        )["paradas"] == 8
    )

    # ----------------------------------------------------------
    # Regra de baldeação
    # ----------------------------------------------------------

    assert (
        pode_baldear(
            True,
            True,
            True,
            False
        ) is False
        and
        pode_baldear(
            True,
            True,
            False,
            False
        ) is True
    )

    # ----------------------------------------------------------
    # Contagem de baldeações
    # ----------------------------------------------------------

    caminho = [
        "Vila Madalena",
        "Sumaré",
        "Clínicas",
        "Consolação",
        "Trianon-Masp",
        "Brigadeiro",
        "Paraíso",
        "Ana Rosa",
        "Vila Mariana"
    ]

    assert contar_baldeacoes(
        caminho,
        LINHAS_DO_TRECHO
    ) == (
        1,
        [("Ana Rosa", "Linha 1-Azul")]
    )

    # ----------------------------------------------------------
    # Intérprete offline
    # ----------------------------------------------------------

    bruto = interpretar_offline(
        "Estou na Catedral da Sé e quero ir ao "
        "Terminal Rodoviário Jabaquara"
    )

    assert resolver_nome(
        bruto["origem"]
    ) == (
        "local",
        "Catedral da Sé"
    )

    assert resolver_nome(
        bruto["destino"]
    ) == (
        "local",
        "Terminal Rodoviário Jabaquara"
    )

    assert resolver_nome(
        "Avenida Paulista"
    ) is None

    assert resolver_nome(
        "Estação Paulista"
    ) is None

    # ----------------------------------------------------------
    # Intérprete offline — horário
    # ----------------------------------------------------------

    bruto = interpretar_offline(
        "Quero sair de Tucuruvi às 18:30 e ir para Jabaquara"
    )

    assert bruto["horario"] == "18:30"

    # ----------------------------------------------------------
    # Narrador offline
    # ----------------------------------------------------------

    texto = narrar_offline(
        viagem(
            "Tucuruvi",
            "Corinthians-Itaquera"
        )
    )

    assert (
        "Na estação Sé, troque para a Linha 3-Vermelha"
        in texto
    )

    # ----------------------------------------------------------
    # Nova R7 no narrador
    # ----------------------------------------------------------

    r = planejar(
        {
            "origem": ("estacao", "Tucuruvi"),
            "destino": ("estacao", "Corinthians-Itaquera"),
            "horario": "08:00"
        },

    )

    texto = narrar_offline(r)

    assert "horário de pico" in texto
    assert "provavelmente cheio" in texto

    # ----------------------------------------------------------
    # R7 — horário noturno
    # ----------------------------------------------------------

    r = planejar(
        {
            "origem": ("estacao", "Tucuruvi"),
            "destino": ("estacao", "Jabaquara"),
            "horario": "22:00",
        },

    )

    assert (
        r["situacao_metro"] == "horário noturno"
    )

    assert (
        r["lotacao_estimada"] == "movimento moderado"
    )

    assert (
        r["horario_utilizado"] == "22:00"
    )

    # ----------------------------------------------------------
    # R7 — horário de abertura
    # ----------------------------------------------------------

    r = planejar(
        {
            "origem": ("estacao", "Tucuruvi"),
            "destino": ("estacao", "Jabaquara"),
            "horario": "05:45",
        },

    )

    assert (
        r["situacao_metro"] == "acabou de abrir"
    )

    assert (
        r["lotacao_estimada"] == "movimento baixo"
    )

    # ----------------------------------------------------------
    # R7 — horário de fechamento
    # ----------------------------------------------------------

    r = planejar(
        {
            "origem": ("estacao", "Tucuruvi"),
            "destino": ("estacao", "Jabaquara"),
            "horario": "23:30",
        },

    )

    assert (
        r["situacao_metro"] == "prestes a fechar"
    )

    assert (
        r["lotacao_estimada"] == "movimento baixo"
    )

    # ----------------------------------------------------------
    # R7 — sem horário informado
    #
    # Deve utilizar o horário do computador.
    # Não verificamos o valor exato porque ele muda conforme
    # o momento em que o teste é executado.
    # ----------------------------------------------------------

    r = viagem(
        "Tucuruvi",
        "Jabaquara"
    )

    assert r.get("horario_utilizado") is not None
    assert isinstance(
        r.get("horario_utilizado"),
        str
    )

    assert len(
        r["horario_utilizado"]
    ) == 5

    assert (
        r["horario_utilizado"][2] == ":"
    )

    pedido_noite = {
        "origem": ("estacao", "Tucuruvi"),
        "destino": ("estacao", "Jabaquara"),
    }

    for h in ("00:00", "03:00", "04:39"):
        r = planejar({**pedido_noite, "horario": h})
        assert r["metro_fechado"] is True, h
        assert r["caminho"] is None and r["paradas"] is None and r["segmentos"] == [], h
        assert r["visitados"] == [], h

    texto = narrar_offline(planejar({**pedido_noite, "horario": "03:00"}))
    assert "fechado" in texto and "04:40" in texto and "03:00" in texto
    assert "Embarque" not in texto

    for h in ("04:40", "12:00", "23:59"):
        r = planejar({**pedido_noite, "horario": h})
        assert r["metro_fechado"] is False and r["caminho"], h

    comp = comparar_algoritmos({**pedido_noite, "horario": "02:00"})
    assert comp["BFS"]["caminho"] is None and comp["DFS"]["caminho"] is None

    RELOGIO_FIXO = "03:00"
    r = planejar(pedido_noite)
    assert r["horario_utilizado"] == "03:00" and r["metro_fechado"] and r["caminho"] is None
    RELOGIO_FIXO = "12:00"

    r = planejar({**pedido_noite, "horario": "03:00"})
    assert not narrador_alucinou(
        "O metrô está fechado às 03:00 e volta a operar às 04:40, "
        "por isso não há rota agora.", r
    )

    r = viagem(
        "Patriarca-Vila Ré",
        "São Judas"
    )

    assert [
        n
        for *_, n in r["segmentos"]
    ] == [
        10,
        10
    ]

    assert r["paradas"] == 20

    assert (
        "siga 10 parada(s) até Sé"
        in narrar_offline(r)
    )

    assert narrador_alucinou(
        "Viaje por 20 estações até a Sé, "
        "onde troque para a Linha 1-Azul.",
        r
    )

    assert not narrador_alucinou(
        "Na Linha 3-Vermelha, 10 paradas até a "
        "estação Sé; na estação Sé, troque para a "
        "Linha 1-Azul e siga 10 paradas até São Judas. "
        "Total: 20 paradas.",
        r
    )

    # ----------------------------------------------------------
    # Horário: extração, validação e fuso de São Paulo
    # ----------------------------------------------------------

    assert extrair_horario("às 18h30") == "18:30"
    assert extrair_horario("às 8 da manhã") == "08:00"
    assert extrair_horario("às 7 da noite") == "19:00"
    assert extrair_horario("às 8h da noite") == "20:00"
    assert extrair_horario("3 da tarde") == "15:00"
    assert extrair_horario("meia-noite e 12 da noite") == "00:00"
    assert extrair_horario("25:00") is None
    assert extrair_horario("pela Linha 3, sem horário") is None

    assert validar_horario("8:05") == "08:05"
    assert validar_horario("18h") == "18:00"
    assert validar_horario("25:00") is None
    assert validar_horario("às 8") is None
    assert validar_horario(None) is None

    try:
        viagem_invalida = planejar({
            "origem": ("estacao", "Tucuruvi"),
            "destino": ("estacao", "Jabaquara"),
            "horario": "25:99",
        })
        assert False, "horário inválido deveria levantar ValueError"
    except ValueError:
        pass

    r = planejar({
        "origem": ("estacao", "Tucuruvi"),
        "destino": ("estacao", "Corinthians-Itaquera"),
        "horario": "18:30",
    })

    assert r["horario_informado"] is True

    assert not narrador_alucinou(
        "Às 18:30, embarque em Tucuruvi na Linha 1-Azul; na estação Sé, "
        "troque para a Linha 3-Vermelha até Corinthians-Itaquera. "
        "Total: 22 paradas.",
        r
    )

    assert motivo_alucinacao(
        "Às 18:30 siga por 99 paradas; na estação Sé, troque de linha.", r
    ) is not None

    # Easter egg

    assert mensagem_torcedor(
        {
            "origem": ("estacao", "Tucuruvi"),
            "destino": ("estacao", "Corinthians-Itaquera")
        }
    )

    assert mensagem_torcedor(
        {
            "origem": ("estacao", "Tucuruvi"),
            "destino": ("local", "Nubank Parque")
        }
    )

    assert mensagem_torcedor(
        {
            "origem": ("estacao", "Tucuruvi"),
            "destino": ("estacao", "Jabaquara")
        }
    ) is None

    # Easter egg só no DESTINO: sair do estádio não dispara a mensagem
    assert mensagem_torcedor(
        {
            "origem": ("local", "Nubank Parque"),
            "destino": ("estacao", "Jabaquara")
        }
    ) is None

    assert mensagem_torcedor(
        {
            "origem": ("estacao", "Corinthians-Itaquera"),
            "destino": ("local", "Nubank Parque")
        }
    ) == EASTER_EGGS_TORCEDOR["Nubank Parque"]

    # "Jogo do Palmeiras" → Barra Funda (nunca Itaquera)
    bruto = interpretar_offline("Estou na Sé e vou ao jogo do Palmeiras")
    assert bruto["origem"] == "Sé" and bruto["destino"] == "Nubank Parque"
    assert resolver_nome("jogo do Palmeiras") == ("local", "Nubank Parque")
    assert resolver_nome("Barra Funda") == ("estacao", "Palmeiras-Barra Funda")

    r = planejar({
        "origem": resolver_nome(bruto["origem"]),
        "destino": resolver_nome(bruto["destino"]),
    })
    assert r["destino"] == "Palmeiras-Barra Funda"

    bruto = interpretar_offline("Quero ir ao jogo do Corinthians saindo do Tucuruvi")
    assert bruto["destino"] == "Neo Química Arena" or bruto["origem"] == "Neo Química Arena"

    # "Palmeiras-Barra Funda" (estação) não pode virar o apelido "Palmeiras"
    assert interpretar_offline("Vou de Tucuruvi até Palmeiras-Barra Funda")["destino"] == "Palmeiras-Barra Funda"

    print("✅ Testes complementares também passaram.")


def rodar_testes():
    """Roda os testes com o relógio fixo e SEMPRE devolve o relógio real no final."""
    global RELOGIO_FIXO
    try:
        _rodar_testes()
    finally:
        RELOGIO_FIXO = None


rodar_testes()

🔧 Sanity check (modelagem): 52 estações, sem vizinhos duplicados, Paraíso–Ana Rosa nas 2 linhas ✅

--- 6 casos obrigatórios do enunciado (BFS) ---
✅ Caso 1 — Tucuruvi → Corinthians-Itaquera
    obtido:   (22, 1, [('Sé', 'Linha 3-Vermelha')])
✅ Caso 2 — Vila Madalena → Jabaquara
    obtido:   (14, 1, [('Ana Rosa', 'Linha 1-Azul')])
✅ Caso 3 — Palmeiras-Barra Funda → Vila Prudente
    obtido:   (16, 2, [('Sé', 'Linha 1-Azul'), ('Ana Rosa', 'Linha 2-Verde')])
✅ Caso 4 — Tucuruvi → Brás, com a Sé fechada
    obtido:   None
✅ Caso 5 — Vila Madalena → Jabaquara, com o Paraíso fechado
    obtido:   None
✅ Caso 6 — Vila Prudente → Jabaquara, com o Paraíso fechado (desvio por Ana Rosa)
    obtido:   (13, True, False)

--- Casos Extras ---
✅ Caso 7 — R6 — integrações são DEDUZIDAS pelo motor (ninguém digitou à mão)
    obtido:   ['Ana Rosa', 'Paraíso', 'Sé']
✅ Caso 8 — R7 — horário informado: 08:00
    obtido:   ('horário de pico', 'provavelmente cheio', '08:00')

8/8 casos passaram.

--- Testes

In [ ]:
# Célula 16 — mostrar o app
try:
    from IPython.display import display
    painel = montar_painel()
    display(painel)
except ImportError:
    print("ipywidgets não encontrado: rode a célula 1 (%pip install ipywidgets) e reinicie o kernel.")

## Declaração de uso de IA
Usei o Claude para revisar o MetrôBot: ele me ajudou a encontrar e corrigir erros ( o horário em UTC no Colab e o horário do painel que não chegava ao planejador), a criar apelidos de locais (ex.: "jogo do Palmeiras" → Nubank Parque).